# 04 — Advanced Tree & Boosting Models

Train and compare:
- Random Forest
- XGBoost
- CatBoost (native categorical features)

All models use a log-price target and the same leakage-safe train/test split.

In [1]:
from pathlib import Path
import sys, json, os

def _find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "bonbanh_usedcar_dataset.csv").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("Project root not found. Keep notebooks inside the CarValue_AI_Complete_Project folder.")

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /mnt/data/CarValue_AI_Complete_Project_v2


In [2]:
import time
import pandas as pd
import numpy as np
from src.carvalue_utils import (
    ensure_modeling_artifacts, get_train_test, TARGET, regression_metrics, save_bundle,
    fit_random_forest_bundle, fit_xgboost_bundle, fit_catboost_bundle
)

model_df, reference_year = ensure_modeling_artifacts(PROJECT_ROOT)
train, test = get_train_test(model_df)
y_test = test[TARGET].to_numpy(float)
print("Train/Test:", train.shape, test.shape)


Train/Test: (18591, 31) (4595, 31)


In [3]:
results = []
predictions = {"row_id": test.index, "actual_price_vnd": y_test}

def train_eval(name, fit_fn, filename):
    t0 = time.perf_counter()
    bundle = fit_fn(train, reference_year)
    seconds = time.perf_counter() - t0
    pred = bundle.predict(test)
    met = regression_metrics(y_test, pred)
    met["Train_seconds"] = seconds
    results.append({"Model": name, **met})
    predictions[name] = pred
    save_bundle(bundle, PROJECT_ROOT / "models" / filename)
    print(f"{name}: MAE={met['MAE_million_VND']:.2f}M | RMSE={met['RMSE_million_VND']:.2f}M | R2={met['R2']:.4f} | {seconds:.1f}s")
    return bundle


In [4]:
rf = train_eval("Random Forest", fit_random_forest_bundle, "random_forest.joblib")


Random Forest: MAE=68.26M | RMSE=202.18M | R2=0.9751 | 1.2s


In [5]:
xgb = train_eval("XGBoost", fit_xgboost_bundle, "xgboost.joblib")


XGBoost: MAE=77.24M | RMSE=247.48M | R2=0.9627 | 0.5s


In [6]:
cat = train_eval("CatBoost", fit_catboost_bundle, "catboost.joblib")


CatBoost: MAE=91.56M | RMSE=266.36M | R2=0.9568 | 6.2s


In [7]:
metrics_df = pd.DataFrame(results).sort_values("MAE_VND").reset_index(drop=True)
pred_df = pd.DataFrame(predictions)
metrics_df.to_csv(PROJECT_ROOT / "artifacts" / "metrics_advanced.csv", index=False)
pred_df.to_csv(PROJECT_ROOT / "artifacts" / "predictions_advanced.csv", index=False)
display(metrics_df.round(4))


,Model,MAE_VND,MAE_million_VND,RMSE_VND,RMSE_million_VND,R2,RMSLE,Train_seconds
0,Random Forest,6.826063e+07,68.2606,2.021760e+08,202.176,0.9751,0.1372,1.1964
1,XGBoost,7.723584e+07,77.2358,2.474840e+08,247.484,0.9627,0.1376,0.5377
2,CatBoost,9.156156e+07,91.5616,2.663580e+08,266.358,0.9568,0.1374,6.2338
